# 02c Estimate: the regression models with indicators (model family 2)

**Purpose:** The second model family: the three regressions of `02b_estimate_regression.ipynb` (linear regression, Ridge, Lasso) with indicators next to the series' own past, in the same backtest and with the same rules, so that any gain or loss can be attributed to the indicators (project plan: one block at a time). The three indicators of `01b` (the routes, the neighbouring countries, the conflict index) enter **one at a time**, **two at a time** and **all together** ("all indicators"); comparing these shows which data lower the error. The lag of the conflict index is chosen inside each training window.
**Input:** `../data/processed/asylum_clean.csv` (from `01_preprocess.ipynb`), `routes_clean.csv`, `neighbours_clean.csv`, `nationality_shares_clean.csv` and `conflict_intensity_clean.csv` (from `01b_preprocess_indicators.ipynb`).
**Output:** `../data/processed/backtest_forecasts_indicator.csv` (same columns as `backtest_forecasts_regression.csv`) , `../output/tables/02c_lasso_kept.csv` and `../output/tables/02c_conflict_lags.csv`.

As before, the test period (2024-10 to 2026-09) is **not used**. We compare the models in `03_evaluate.ipynb`.

In [1]:
import numpy as np
import pandas as pd
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

np.random.seed(42)

## 1. Load the target and the indicators

We put the log of each indicator next to the target, month by month. `reindex` puts each indicator on the months of the target; months an indicator does not cover stay `NaN` (the routes start in January 2009, the target in January 2008).

*Not seen in class:* `reindex(months)` puts a series on a given list of row labels and leaves `NaN` where a label is missing.

In [2]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")
y = clean["applicants_total"].to_numpy(dtype=float)
log_y = clean["log_applicants"].to_numpy(dtype=float)
months = clean.index

def load(path):
    data = pd.read_csv(path)
    data["month"] = pd.to_datetime(data["month"])
    return data.set_index("month")

log_routes = load("../data/processed/routes_clean.csv")["log_routes"].reindex(months).to_numpy(dtype=float)
log_neighbours = load("../data/processed/neighbours_clean.csv")["log_neighbours"].reindex(months).to_numpy(dtype=float)

FIRST_ORIGIN = pd.Timestamp("2012-12-01")
LAST_BACKTEST = pd.Timestamp("2024-09-01")
H_MAX = 3
first_origin = int(np.where(months == FIRST_ORIGIN)[0][0])
last_backtest = int(np.where(months == LAST_BACKTEST)[0][0])
shares = load("../data/processed/nationality_shares_clean.csv").reindex(months)
intensity = load("../data/processed/conflict_intensity_clean.csv")

def conflict_change(t, L, k):
    # change of log(1 + deaths) between months t-L-k and t-L, weighted by the top 10 nationalities at origin t
    w = shares.iloc[t]
    if w.isna().all():
        return np.nan
    top = w.nlargest(10)
    m1 = months[t] - pd.DateOffset(months=L)
    m0 = months[t] - pd.DateOffset(months=L + k)
    if m1 not in intensity.index or m0 not in intensity.index:
        return np.nan
    weighted = top * (intensity.loc[m1, top.index] - intensity.loc[m0, top.index])
    return float(weighted.sum()) if weighted.notna().all() else np.nan    # NaN if a month is not in UCDP

LAGS = range(1, 13)                                   # the conflict months may end 1 to 12 months before the origin
conflict = {(L, k): np.array([conflict_change(t, L, k) for t in range(len(months))]) for L in LAGS for k in [3, 6]}

print("Missing months: routes", np.isnan(log_routes).sum(), "| neighbours", np.isnan(log_neighbours).sum())

Missing months: routes 12 | neighbours 1


## 2. The indicator features

`01b` fixed the rule: **at a forecast origin $t$, the newest month we may use is $t-1$** for both indicators (both are published about one month later than the target). We decided on two features per indicator before running any model, with $R$ = detections on the three routes and $N$ = applications in DE + AT + FR + IT:

| Feature | Formula | Idea |
|---|---|---|
| `routes_change_last_month` | $\log R_{t-1} - \log R_{t-2}$ | the newest monthly change on the routes |
| `routes_change_3_months` | $\log R_{t-1} - \log R_{t-4}$ | the change over the last three known months |
| `neighbours_change_last_month` | $\log N_{t-1} - \log N_{t-2}$ | the newest monthly change in the neighbouring countries |
| `neighbours_change_3_months` | $\log N_{t-1} - \log N_{t-4}$ | the change over the last three known months |
| `conflict_3_months` | $\text{conflict}_t(L, 3)$ (`01b`, section 13) | the change of conflict deaths over 3 months, ending $L$ months before the origin |
| `conflict_6_months` | $\text{conflict}_t(L, 6)$ | the same over 6 months |

They are **changes**, like the other features, for the reason of notebook 200 (trending levels can look related without any link). We did not use a change over a whole year: it would need the indicator 13 months back, and the first training windows would become very small.

**The lag $L$ of the conflict index is chosen inside each training window**, like `alpha` in `02b`: at every origin and horizon we try $L = 1, \dots, 12$, fit the linear regression on the training rows and keep the lag with the lowest MSE on the same two validation blocks of 12 months; Ridge and Lasso then use that lag and choose their `alpha` as usual. So the lag is chosen only with months known at the origin, and the backtest still tests the whole procedure out of sample. We choose the lag with the linear regression (no penalty to tune), once per origin and horizon, to keep the run short.

Seven combinations: each indicator alone, each pair, and all three ("all indicators"); the pairs are "all indicators but one", which shows how much each indicator adds when the others are in. All other features, the target, the models, the grids of `alpha` and the validation inside each training window are **exactly those of `02b_estimate_regression.ipynb`**.

In [3]:
features = {
    "routes": {
        "routes_change_last_month": lambda t: log_routes[t - 1] - log_routes[t - 2],
        "routes_change_3_months": lambda t: log_routes[t - 1] - log_routes[t - 4],
    },
    "neighbours": {
        "neighbours_change_last_month": lambda t: log_neighbours[t - 1] - log_neighbours[t - 2],
        "neighbours_change_3_months": lambda t: log_neighbours[t - 1] - log_neighbours[t - 4],
    },
}

def block_features(block, L):
    # the features of one indicator; for the conflict index they depend on the lag L
    if block == "conflict":
        return {"conflict_3_months": lambda t: conflict[(L, 3)][t], "conflict_6_months": lambda t: conflict[(L, 6)][t]}
    return features[block]

indicators = ["routes", "neighbours", "conflict"]
combinations = {"routes": ["routes"], "neighbours": ["neighbours"], "conflict": ["conflict"],
                "routes + neighbours": ["routes", "neighbours"], "routes + conflict": ["routes", "conflict"],
                "neighbours + conflict": ["neighbours", "conflict"], "all indicators": indicators}

def make_table(h, combination, L=None):
    # Same table as in 02b, plus the features of the indicators (months known at the origin only)
    extra_features = {k: f for b in combinations[combination] for k, f in block_features(b, L).items()}
    rows = []
    for t in range(12, len(log_y) - h):
        values = {name: f(t) for name, f in extra_features.items()}
        if any(np.isnan(v) for v in values.values()):     # an indicator starts when all its months exist
            continue
        rows.append({
            "t": t,
            "change_last_month": log_y[t] - log_y[t - 1],
            "change_month_before": log_y[t - 1] - log_y[t - 2],
            "same_change_last_year": log_y[t + h - 12] - log_y[t - 12],
            **values,
            "calendar_month": months[t + h].month,
            "target": log_y[t + h] - log_y[t],
        })
    table = pd.DataFrame(rows).set_index("t")
    dummies = pd.get_dummies(table["calendar_month"], prefix="month", drop_first=True).astype(float)
    names = ["change_last_month", "change_month_before", "same_change_last_year"] + list(extra_features)
    X = pd.concat([table[names], dummies], axis=1)
    return X, table["target"]

tables = {}
for comb, parts in combinations.items():
    for h in range(1, H_MAX + 1):
        for L in (LAGS if "conflict" in parts else [None]):
            tables[(comb, h, L)] = make_table(h, comb, L)
for comb, parts in combinations.items():
    X2, _ = tables[(comb, 2, 1 if "conflict" in parts else None)]
    print(f"{comb:22s} h = 2: {X2.shape[0]} rows, {X2.shape[1]} features, first origin {months[X2.index.min()]:%Y-%m}, "
          f"missing values {X2.isna().sum().sum()}")
print("Tables:", len(tables))
tables[("all indicators", 2, 1)][0][["routes_change_last_month", "neighbours_change_last_month", "conflict_3_months", "conflict_6_months"]].describe().round(3)

routes                 h = 2: 207 rows, 16 features, first origin 2009-05, missing values 0
neighbours             h = 2: 211 rows, 16 features, first origin 2009-01, missing values 0
conflict               h = 2: 205 rows, 16 features, first origin 2009-01, missing values 0
routes + neighbours    h = 2: 207 rows, 18 features, first origin 2009-05, missing values 0
routes + conflict      h = 2: 201 rows, 18 features, first origin 2009-05, missing values 0
neighbours + conflict  h = 2: 205 rows, 18 features, first origin 2009-01, missing values 0
all indicators         h = 2: 201 rows, 20 features, first origin 2009-05, missing values 0
Tables: 153


,routes_change_last_month,neighbours_change_last_month,conflict_3_months,conflict_6_months
count,201.000,201.000,201.000,201.000
mean,0.003,0.006,-0.023,-0.053
std,0.376,0.163,0.284,0.370
min,-1.645,-0.919,-1.289,-1.594
25%,-0.231,-0.084,-0.147,-0.186
50%,0.037,0.009,-0.009,-0.003
75%,0.220,0.117,0.121,0.139
max,1.227,0.477,0.845,1.156


**What we see.** For $h = 2$, the tables with one indicator have 16 features (the three own changes, two indicator features, 11 month dummies), those with two indicators 18 and the table with all three 20. No value is missing. The tables with the routes start at the origin May 2009 (the routes start in January 2009 and the 3-month change needs $t-4$), the others in January 2009, like `02b`. With 12 lags for the conflict index, there are 153 tables in all. The route changes move the most (standard deviation 0.38 for the monthly change), the neighbour changes the least (0.16).

## 3. Models, grids and the choice of `alpha`: the same as in `02b`

The functions below are copied from `02b_estimate_regression.ipynb` (only split into two functions, so that the same validation also chooses the lag of the conflict index) (scaling inside the pipeline, two validation blocks of 12 months inside each training window, the same grids).

In [4]:
def make_model(name, alpha=None):
    if name == "Linear regression":
        return make_pipeline(StandardScaler(), LinearRegression())
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    return make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))

alpha_grids = {
    "Linear regression": [None],
    "Ridge": [0.1, 1, 10, 100, 1000, 10000],
    "Lasso": [0.0001, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3],
}

def validation_mse(name, alpha, X, target, T, h, n_blocks=2, block=12):
    # mean MSE on the last two blocks of 12 months before the origin T, each fitted on the rows before it
    first_row, last_row = X.index.min(), T - h
    mse_blocks = []
    for b in range(n_blocks):
        start = last_row - n_blocks * block + 1 + b * block
        valid_rows = list(range(start, start + block))
        train_rows = list(range(first_row, start - h + 1))
        model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
        errors = target.loc[valid_rows].to_numpy() - model.predict(X.loc[valid_rows])
        mse_blocks.append(np.mean(errors ** 2))
    return np.mean(mse_blocks)

def choose_alpha(name, X, target, T, h):
    grid = alpha_grids[name]
    if len(grid) == 1:
        return grid[0]
    return grid[int(np.argmin([validation_mse(name, alpha, X, target, T, h) for alpha in grid]))]

def choose_lag(comb, T, h):
    # the lag of the conflict index with the lowest validation MSE of the linear regression; only lags whose
    # months exist at the origin T can be chosen
    scores = {}
    for L in LAGS:
        X, target = tables[(comb, h, L)]
        if T in X.index:
            scores[L] = validation_mse("Linear regression", None, X, target, T, h)
    return min(scores, key=scores.get)

## 4. The backtest loop

The same loop as in `02b`, once per combination: the same 141 origins and horizons. We check again that no training target lies after the origin (`assert`). For the combinations with the conflict index, the lag is chosen first. This takes about nine minutes.

In [5]:
rows = []
for comb, parts in combinations.items():
    for t in range(first_origin, last_backtest):
        for h in range(1, H_MAX + 1):
            target_pos = t + h
            if target_pos > last_backtest:
                continue
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                L = choose_lag(comb, t, h) if "conflict" in parts else None
                X, target = tables[(comb, h, L)]
                train_rows = list(range(X.index.min(), t - h + 1))
                assert max(train_rows) + h <= t                    # no training target after the origin
                for name in alpha_grids:
                    alpha = choose_alpha(name, X, target, t, h)
                    model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
                    forecast_log = log_y[t] + model.predict(X.loc[[t]])[0]
                    rows.append({
                        "model": f"{name} + {comb}", "origin": months[t], "month": months[target_pos],
                        "h": h, "forecast_log": forecast_log, "actual_log": log_y[target_pos],
                        "forecast": np.exp(forecast_log), "actual": y[target_pos], "alpha": alpha, "lag": L,
                    })

forecasts = pd.DataFrame(rows)
print("Number of stored forecasts:", forecasts.shape[0])

Number of stored forecasts: 8820


## 5. Checks on the stored forecasts

In [6]:
print(forecasts.groupby(["model", "h"]).size().unstack())
print()
print("All forecast months in the backtest (before 2024-10):", (forecasts["month"] < pd.Timestamp("2024-10-01")).all())
print("All forecast months after their origin:", (forecasts["month"] > forecasts["origin"]).all())
print("Missing or infinite forecasts:", (~np.isfinite(forecasts["forecast_log"])).sum())

h                                            1    2    3
model                                                   
Lasso + all indicators                     141  140  139
Lasso + conflict                           141  140  139
Lasso + neighbours                         141  140  139
Lasso + neighbours + conflict              141  140  139
Lasso + routes                             141  140  139
Lasso + routes + conflict                  141  140  139
Lasso + routes + neighbours                141  140  139
Linear regression + all indicators         141  140  139
Linear regression + conflict               141  140  139
Linear regression + neighbours             141  140  139
Linear regression + neighbours + conflict  141  140  139
Linear regression + routes                 141  140  139
Linear regression + routes + conflict      141  140  139
Linear regression + routes + neighbours    141  140  139
Ridge + all indicators                     141  140  139
Ridge + conflict               

**What we see.** As in `02b`: each of the 21 models has 141, 140 and 139 forecasts for $h = 1, 2, 3$, 8,820 together, all in the backtest, all after their origin, none missing. With the `assert` in the loop, and the lag chosen only from the training window, this shows that the indicators used only months known at each origin.

### Which penalties were chosen?

As in `02b` (notebook 209), we check that the chosen `alpha` does not sit mostly at an edge of the grid.

And which lags of the conflict index were chosen?

In [7]:
for name in ["Ridge", "Lasso"]:
    chosen = forecasts[forecasts["model"].str.startswith(name + " +")]
    print(f"{name}: how often each alpha was chosen")
    print(chosen.groupby(["model", "alpha"]).size().unstack(fill_value=0).to_string())
    print()

linear = forecasts[forecasts["model"].str.startswith("Linear regression +") & forecasts["lag"].notna()].copy()
linear["combination"] = linear["model"].str.replace("Linear regression + ", "")
conflict_lags = linear.groupby(["combination", "h", "lag"]).size().unstack(fill_value=0)
conflict_lags.columns = [f"lag {int(L)}" for L in conflict_lags.columns]
conflict_lags.to_csv("../output/tables/02c_conflict_lags.csv")
print("How often each lag of the conflict index was chosen (per origin):")
conflict_lags

Ridge: how often each alpha was chosen
alpha                          0.1      1.0      10.0     100.0    1000.0   10000.0
model                                                                              
Ridge + all indicators              58       24       81      183       56       18
Ridge + conflict                    87       12       84      179       53        5
Ridge + neighbours                  63       25       72      174       56       30
Ridge + neighbours + conflict       47       26       97      170       54       26
Ridge + routes                      92        9       50      196       65        8
Ridge + routes + conflict           97       19       61      179       55        9
Ridge + routes + neighbours         68       17       63      195       59       18

Lasso: how often each alpha was chosen
alpha                          0.0001  0.0010  0.0030  0.0100  0.0300  0.1000  0.3000
model                                                                          

lag 1  lag 2  lag 3  lag 4  lag 5  lag 6  lag 7  \
combination           h                                                    
all indicators        1     24     40      2      4     15      2      9   
                      2      1     25      4      6     12      8     12   
                      3      1     38      9      4      1     12     14   
conflict              1     29     23      1      2     22      9     22   
                      2      4     13      4     19     18     15      9   
                      3      4     15     27      4      4      4      7   
neighbours + conflict 1     31     43      1      2     20      2     10   
                      2      6     25     13      6     13     23      9   
                      3      0     38     17      8      0      4     16   
routes + conflict     1     30     22      5      2     22      5     27   
                      2      1     19      8     15     16      3     19   
                      3      6     20     24      0      1     13     17   

                         lag 8  lag 9  lag 10  lag 11  lag 12  
combination           h                                        
all indicators        1      5      2       8      18      12  
                      2      7     13       6      17      29  
                      3      1      6      22      22       9  
conflict              1      2      1       6      15       9  
                      2      3     11       2      16      26  
                      3      0      3      15      44      12  
neighbours + conflict 1      8      5       4       7       8  
                      2      4     10       3      13      15  
                      3      1      1      12      34       8  
routes + conflict     1      5      0       3      13       7  
                      2      1     10       3      20      25  
                      3      0      6      19      26       7

**What we see.**
- Ridge most often chose `alpha = 100` (170 to 196 of 420 times); the largest value (10,000) was chosen at most 30 times. Lasso most often chose 0.01 or 0.03; the smallest value (0.0001, almost no penalty) up to 128 times.
- **The lag of the conflict index jumps around:** every lag from 1 to 12 is chosen at some origins. For the model with all indicators, lag 2 is chosen most often at $h = 1$ and $h = 3$ (40 and 38 of about 140 origins), lag 12 at $h = 2$ (29); for the conflict index alone at $h = 3$, lag 11 (44 of 139).

**What this means for us.** The grids are wide enough. No lag of the conflict index is chosen at most origins: as in `02d` for the other indicators, the best lag changes from one window to the next, which is what we expect when an indicator has no stable lead.

## 6. What do the models do with the indicators?

The coefficients for $h = 2$ at the last origin (August 2024), on standardised features (notebook 105), for the models with all indicators. A coefficient says what helps to *predict*, not what *causes*.

In [8]:
t = last_backtest - 1
h = 2
L = choose_lag("all indicators", t, h)
X, target = tables[("all indicators", h, L)]
train_rows = list(range(X.index.min(), t - h + 1))

coefficients = pd.DataFrame(index=X.columns)
for name in alpha_grids:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha(name, X, target, t, h)
        model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
    coefficients[f"{name} (alpha = {alpha})"] = model[-1].coef_

print("Origin:", months[t], "| horizon h =", h, "| conflict lag:", L, "| training rows:", len(train_rows))
coefficients.iloc[:9].round(3)

Origin: 2024-08-01 00:00:00 | horizon h = 2 | conflict lag: 11 | training rows: 182


,Linear regression (alpha = None),Ridge (alpha = 0.1),Lasso (alpha = 0.0001)
change_last_month,-0.021,-0.021,-0.021
change_month_before,-0.010,-0.010,-0.010
same_change_last_year,0.002,0.003,0.003
routes_change_last_month,0.015,0.015,0.014
routes_change_3_months,-0.034,-0.034,-0.034
neighbours_change_last_month,0.003,0.003,0.003
neighbours_change_3_months,-0.044,-0.044,-0.044
conflict_3_months,-0.050,-0.050,-0.050
conflict_6_months,0.064,0.064,0.064


Lasso can set a coefficient to exactly 0 (notebook 105). At every origin of the backtest ($h = 2$, all indicators), we record which indicator features it keeps: a feature kept at most origins carries information that the series' own past does not.

In [9]:
indicator_names = ["routes_change_last_month", "routes_change_3_months", "neighbours_change_last_month",
                   "neighbours_change_3_months", "conflict_3_months", "conflict_6_months"]
kept = []
for t in range(first_origin, last_backtest - 1):          # all origins with a known h = 2 target
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        L = choose_lag("all indicators", t, 2)
        X, target = tables[("all indicators", 2, L)]
        train_rows = list(range(X.index.min(), t - 2 + 1))
        alpha = choose_alpha("Lasso", X, target, t, 2)
        model = make_model("Lasso", alpha).fit(X.loc[train_rows], target.loc[train_rows])
    kept.append(pd.Series(model[-1].coef_, index=X.columns)[indicator_names])
kept = pd.DataFrame(kept)
lasso_kept = pd.DataFrame({"indicator": [n.split("_")[0] for n in indicator_names],
                           "share of origins kept": (kept != 0).mean().round(2),
                           "mean coefficient": kept.mean().round(3)})
lasso_kept.index.name = "feature"
lasso_kept.to_csv("../output/tables/02c_lasso_kept.csv")
print("Origins:", len(kept))
lasso_kept

Origins: 140


,indicator,share of origins kept,mean coefficient
feature,,,
routes_change_last_month,routes,0.79,0.012
routes_change_3_months,routes,0.42,-0.011
neighbours_change_last_month,neighbours,0.36,0.002
neighbours_change_3_months,neighbours,0.54,-0.022
conflict_3_months,conflict,0.46,0.002
conflict_6_months,conflict,0.52,-0.001


**What we see.**
- At the last origin (182 training rows), the chosen lag of the conflict index is 11, and Ridge chose a weak penalty (`alpha = 0.1`), so the three models are almost the same. The largest indicator coefficients are the two conflict features, with opposite signs (+0.064 over 6 months, −0.050 over 3 months), then the 3-month changes of the neighbours (−0.044) and of the routes (−0.034). Opposite signs within one indicator mean that no sign should be read on its own.
- Over all 140 origins, Lasso keeps the **monthly route change** at 79% of the origins; the conflict features at 46% and 52%, the neighbours at 36% and 54%, the 3-month route change at 42%.

**What this means for us.** Only the newest route month is kept at most origins. Whether the indicators lower the *forecast errors* is measured in `03_evaluate.ipynb` (section 7).

## 7. Save the forecasts

In [10]:
forecasts.to_csv("../data/processed/backtest_forecasts_indicator.csv", index=False)
print("Saved", forecasts.shape[0], "forecasts with the columns", list(forecasts.columns))

Saved 8820 forecasts with the columns ['model', 'origin', 'month', 'h', 'forecast_log', 'actual_log', 'forecast', 'actual', 'alpha', 'lag']


## Summary

- **Models:** the three regressions of `02b` with the same features, target, grids and validation, plus two features per indicator: each of the routes, the neighbouring countries and the conflict index alone, each pair, and all three ("all indicators"). The routes and the neighbours end at month $t-1$; the lag of the conflict index (1 to 12 months) is chosen inside each training window with the same validation blocks.
- **Checks:** 8,820 forecasts, all in the backtest, after their origin, none missing; the chosen `alpha` lies mostly inside the grids.
- **What the models do:** the chosen conflict lag jumps between all 12 lags; Lasso keeps only the monthly route change at most origins (79%).
- **Output:** `../data/processed/backtest_forecasts_indicator.csv`, `../output/tables/02c_lasso_kept.csv` and `02c_conflict_lags.csv`, compared in `03_evaluate.ipynb`.